# LangChain Hands-on 08 — Embeddings
**Trainer:** Ajeetkumar

**What we will practise**
- What is an embedding? (text → numbers)
- Free HuggingFace embeddings
- embed_query vs embed_documents
- Cosine similarity by hand
- OpenAI / Google / Ollama embeddings
- Caching embeddings

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-huggingface sentence-transformers

# What is an Embedding?
- An embedding converts **text into a list of numbers (vector)** that captures meaning.
- Similar meaning → vectors close to each other.

```
"lost credit card"   → [0.12, -0.33, 0.85, ...]   ┐ close
"stolen debit card"  → [0.10, -0.30, 0.80, ...]   ┘
"biryani recipe"     → [-0.70, 0.45, 0.02, ...]     far
```
- Used for **semantic search, RAG, recommendations, clustering**.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")   # free, runs on Colab

## 1. embed_query — one text

In [ ]:
vector = embeddings.embed_query("How do I block my lost card?")
len(vector)

In [ ]:
vector[:10]

## 2. embed_documents — many texts

In [ ]:
sentences = ["lost credit card", "stolen debit card", "best biryani in Hyderabad", "UPI payment failed"]
vectors = embeddings.embed_documents(sentences)
len(vectors), len(vectors[0])

## 3. Cosine similarity — how close are two meanings?

In [ ]:
import numpy as np

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        print(f"{sentences[i]:<28} vs {sentences[j]:<28} → {cosine(vectors[i], vectors[j]):.3f}")

## 4. Mini semantic search (without a vector DB)

In [ ]:
faq = ["How to reset my net banking password?",
       "What is the daily UPI limit?",
       "How to block a stolen card?",
       "How to update my mobile number?",
       "What are the home loan interest rates?"]
faq_vectors = embeddings.embed_documents(faq)

In [ ]:
def search(question):
    q = embeddings.embed_query(question)
    scores = [cosine(q, v) for v in faq_vectors]
    best = int(np.argmax(scores))
    return faq[best], round(scores[best], 3)

In [ ]:
search("someone took my debit card")

In [ ]:
search("forgot my login password")

## 5. Other embedding providers (same methods!)
```python
# OpenAI       ! pip install langchain-openai
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")        # 1536 dims

# Google       ! pip install langchain-google-genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Ollama (local) ! pip install langchain-ollama
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# One function for all
from langchain.embeddings import init_embeddings
embeddings = init_embeddings("openai:text-embedding-3-small")
```

## 6. Cache embeddings — don't pay twice for the same text

In [ ]:
! pip install -q langchain-classic

In [ ]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

In [ ]:
store = LocalFileStore("./emb_cache/")
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, store, namespace="minilm")

In [ ]:
import time
t = time.time(); cached_embeddings.embed_documents(faq); print("first :", round(time.time() - t, 3), "s")
t = time.time(); cached_embeddings.embed_documents(faq); print("cached:", round(time.time() - t, 3), "s")

# Choosing an embedding model
| Model | Cost | Dims | Note |
|---|---|---|---|
| all-MiniLM-L6-v2 | free | 384 | fast, good for learning |
| bge / e5 (HuggingFace) | free | 768–1024 | better quality |
| OpenAI text-embedding-3-small | paid, cheap | 1536 | strong, popular |
| Gemini embedding | paid / free tier | 3072 | multilingual |

⚠️ Use the **same** embedding model for documents and queries.

## 🧪 Practice
1. Embed 10 movie plots and find the closest one to "a robot falls in love".
2. Compare similarity of the Hindi and English versions of the same sentence.
3. Try `BAAI/bge-small-en-v1.5` and compare scores with MiniLM.